In [ ]:
# Importe
from pathlib import Path
import json
import math
import platform
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
try:
    import pm4py
except ImportError as exc:
    raise ImportError('PM4Py fehlt. Bitte in der .venv installieren: pip install -U pm4py') from exc
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, brier_score_loss, confusion_matrix, f1_score, log_loss, precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
print('Notebook 09 läuft.')
print('Python:', platform.python_version())
print('Platform:', platform.platform())


In [ ]:
# Konfiguration
PROJECT_ROOT = Path('..').resolve()
DATA_RAW = PROJECT_ROOT / 'data_raw'
LOG_PATH = DATA_RAW / 'BPI_Challenge_2018.xes.gz'
RECON_CORE_CANDIDATES = [PROJECT_ROOT / 'outputs' / 'benchmark_reconciliation_two_perspectives' / 'tables' / '19_case_level_reconciliation_core.csv', PROJECT_ROOT / 'outputs' / 'benchmark_reconciliation_two_perspectives' / '19_case_level_reconciliation_core.csv']
RECON_CORE_PATH = next((p for p in RECON_CORE_CANDIDATES if p.exists()), None)
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'comparative_prediction_final_robustness'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
for directory in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
N_JOBS = -1
TRAIN_SHARE_WITHIN_2015 = 0.75
MIN_POS_TRAIN = 30
MIN_POS_VALIDATION = 10
MIN_POS_TEST = 20
BOOTSTRAP_REPETITIONS = 500
RUN_2017_SENSITIVITY = True
RUN_PERMUTATION_IMPORTANCE = True
PERMUTATION_SAMPLE_N = 2500
PERMUTATION_REPEATS = 5
TOP_N_RAW_ACTIVITIES = 30
TOP_N_COMBINED_ACTIVITIES = 50
TOP_N_SUBPROCESSES = 15
TOP_N_DOCTYPES = 12
TOP_N_RESOURCES = 25
MIN_CASES_PER_CATEGORY = 40
RF_N_ESTIMATORS = 180
RF_MIN_SAMPLES_LEAF = 5
RF_MAX_DEPTH = None
PREFIX_SPECS = [{'prefix_id': 'static_first_event', 'prefix_type': 'event', 'value': 1, 'information_rank': 1}, {'prefix_id': 'event20', 'prefix_type': 'event', 'value': 20, 'information_rank': 20}, {'prefix_id': 'time30d', 'prefix_type': 'time_days', 'value': 30, 'information_rank': 30}, {'prefix_id': 'time60d', 'prefix_type': 'time_days', 'value': 60, 'information_rank': 60}, {'prefix_id': 'time90d', 'prefix_type': 'time_days', 'value': 90, 'information_rank': 90}, {'prefix_id': 'time120d', 'prefix_type': 'time_days', 'value': 120, 'information_rank': 120}]
MODEL_NAMES = ['dummy_prior', 'logreg_balanced', 'rf_balanced']
TARGET_CONFIG = {'label_scd_p90_or_global': {'short_name': 'original_scd', 'description': 'Gesamte operative strukturelle Komplexität inklusive Inspection-Aufwand.', 'primary_scenario': 'observed_prefix', 'sensitivity_scenarios': ['no_inspection_context'], 'years_primary': ['2015', '2016'], 'allow_2017_sensitivity': True}, 'label_scd_no_inspection_p90_or': {'short_name': 'residual_scd', 'description': 'Komplexität außerhalb des Inspection-Subprozesses.', 'primary_scenario': 'inspection_blind_residual', 'sensitivity_scenarios': ['observed_prefix'], 'years_primary': ['2015', '2016'], 'allow_2017_sensitivity': True}, 'label_reopened_official': {'short_name': 'reopened', 'description': 'Offizieller Change-/Objection-Benchmark.', 'primary_scenario': 'reopened_at_risk', 'sensitivity_scenarios': [], 'years_primary': ['2015', '2016'], 'allow_2017_sensitivity': False}, 'label_late_payment_official_stable': {'short_name': 'late_strict', 'description': 'Strenge offizielle Late-Payment-Variante; nur Sensitivität.', 'primary_scenario': 'observed_prefix', 'sensitivity_scenarios': [], 'years_primary': ['2015', '2016'], 'allow_2017_sensitivity': False, 'optional': True}}
SCENARIO_REGISTRY = {'observed_prefix': {'description': 'Alle im Prefix tatsächlich beobachteten Events; keine finalen Case-Metriken und keine Inspection-Selection-Flags.', 'remove_inspection_events': False, 'remove_reopened_target_events': False, 'at_risk_reopened': False, 'drop_feature_patterns': ['selected_random', 'selected_risk', 'selected_manually', 'selected_any_inspection']}, 'no_inspection_context': {'description': 'Inspection-Events und direkte Inspection-Merkmale werden aus dem Feature-Set entfernt.', 'remove_inspection_events': True, 'remove_reopened_target_events': False, 'at_risk_reopened': False, 'drop_feature_patterns': ['inspection', 'on_site', 'onsite', 'selected_random', 'selected_risk', 'selected_manually', 'selected_any_inspection']}, 'inspection_blind_residual': {'description': 'Primärdesign für Residual SCD; Prefix-Zeitfenster wird auf Originaltimeline definiert, danach werden Inspection-Events entfernt.', 'remove_inspection_events': True, 'remove_reopened_target_events': False, 'at_risk_reopened': False, 'drop_feature_patterns': ['inspection', 'on_site', 'onsite', 'selected_random', 'selected_risk', 'selected_manually', 'selected_any_inspection']}, 'reopened_at_risk': {'description': 'Change-/Objection-Events sind keine Features; positive Fälle werden ausgeschlossen, sobald Reopening bereits vor dem Prefix-Cutoff beobachtet wurde.', 'remove_inspection_events': False, 'remove_reopened_target_events': True, 'at_risk_reopened': True, 'drop_feature_patterns': ['change', 'objection', 'selected_random', 'selected_risk', 'selected_manually', 'selected_any_inspection']}}
if not LOG_PATH.exists():
    xes_candidates = sorted(DATA_RAW.glob('*.xes*')) + sorted(DATA_RAW.glob('**/*.xes*'))
    if not xes_candidates:
        raise FileNotFoundError(f'Keine XES-Datei in {DATA_RAW} gefunden.')
    LOG_PATH = xes_candidates[0]
if RECON_CORE_PATH is None:
    raise FileNotFoundError('Die Reconciliation-Core-Datei wurde nicht gefunden. Bitte Notebook 08 vollständig ausführen. Erwartet wird 19_case_level_reconciliation_core.csv unter outputs/benchmark_reconciliation_two_perspectives/.')
print('Project root:', PROJECT_ROOT)
print('Log path:', LOG_PATH)
print('Reconciliation core:', RECON_CORE_PATH)
print('Output root:', OUTPUT_ROOT)
print('Targets:', list(TARGET_CONFIG))
print('Prefixes:', [p['prefix_id'] for p in PREFIX_SPECS])


In [ ]:
# Hilfsfunktionen
created_tables = []
created_figures = []
analysis_notes = []

def save_csv(obj, filename, index=True):
    path = TABLE_DIR / filename
    if isinstance(obj, pd.Series):
        obj.to_frame().to_csv(path, index=index, encoding='utf-8-sig')
    else:
        obj.to_csv(path, index=index, encoding='utf-8-sig')
    created_tables.append(path)
    return path

def save_json(obj, filename):
    path = TABLE_DIR / filename
    with open(path, 'w', encoding='utf-8') as handle:
        json.dump(obj, handle, indent=2, ensure_ascii=False, default=str)
    created_tables.append(path)
    return path

def save_fig(fig, filename):
    path = FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    created_figures.append(path)
    return path

def robust_to_bool(series, default=False):
    if isinstance(series, pd.Series):
        if pd.api.types.is_bool_dtype(series):
            return series.fillna(default).astype(bool)
        text = series.astype(str).str.strip().str.lower()
        true_values = {'true', '1', '1.0', 'yes', 'y', 'ja', 'wahr', 't'}
        false_values = {'false', '0', '0.0', 'no', 'n', 'nein', 'falsch', 'nan', 'none', '<na>', '', 'f'}
        out = pd.Series(bool(default), index=series.index)
        out[text.isin(true_values)] = True
        out[text.isin(false_values)] = False
        numeric = pd.to_numeric(series, errors='coerce')
        out[numeric.fillna(0) > 0] = True
        return out.astype(bool)
    if pd.isna(series):
        return bool(default)
    return str(series).strip().lower() in {'true', '1', '1.0', 'yes', 'y', 'ja', 'wahr', 't'}

def sanitize_col_name(value):
    text = re.sub('[^0-9a-zA-ZäöüÄÖÜß]+', '_', str(value).strip())
    text = re.sub('_+', '_', text).strip('_')
    return (text or 'missing')[:90]

def make_ohe():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=MIN_CASES_PER_CATEGORY)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

def safe_metric(metric_func, y_true, y_score):
    try:
        y_true = np.asarray(y_true).astype(int)
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(metric_func(y_true, y_score))
    except Exception:
        return np.nan

def safe_log_loss(y_true, y_score):
    try:
        y_true = np.asarray(y_true).astype(int)
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(log_loss(y_true, y_score, labels=[0, 1]))
    except Exception:
        return np.nan

def select_threshold_by_f1(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return (0.5, np.nan)
    thresholds = np.unique(np.quantile(y_score, np.linspace(0.01, 0.99, 99)))
    best_threshold, best_f1 = (0.5, -1.0)
    for threshold in thresholds:
        value = f1_score(y_true, y_score >= threshold, zero_division=0)
        if value > best_f1:
            best_threshold, best_f1 = (float(threshold), float(value))
    return (best_threshold, best_f1)

def evaluate_scores(y_true, y_score, threshold, metadata):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = y_score >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    row = dict(metadata)
    row.update({'threshold': float(threshold), 'n': int(len(y_true)), 'positive_cases': int(y_true.sum()), 'prevalence_pct': float(y_true.mean() * 100) if len(y_true) else np.nan, 'roc_auc': safe_metric(roc_auc_score, y_true, y_score), 'pr_auc_average_precision': safe_metric(average_precision_score, y_true, y_score), 'precision': float(precision_score(y_true, y_pred, zero_division=0)), 'recall': float(recall_score(y_true, y_pred, zero_division=0)), 'f1': float(f1_score(y_true, y_pred, zero_division=0)), 'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)), 'accuracy': float(accuracy_score(y_true, y_pred)), 'brier_score': float(brier_score_loss(y_true, y_score)) if len(y_true) else np.nan, 'log_loss': safe_log_loss(y_true, y_score), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)})
    return row

def top_values(prefix_events, column, train_cases, top_n):
    if column is None or column not in prefix_events.columns or len(prefix_events) == 0:
        return []
    subset = prefix_events[prefix_events[CASE_COL].isin(train_cases)]
    return subset[column].astype(str).value_counts(dropna=False).head(top_n).index.tolist()

def add_count_features(base, events, column, values, feature_prefix):
    if column is None or column not in events.columns or (not values):
        return base
    allowed = {str(value) for value in values}
    subset = events[events[column].astype(str).isin(allowed)]
    if len(subset) == 0:
        for value in values:
            base[f'cnt_{feature_prefix}__{sanitize_col_name(value)}'] = 0
        return base
    counts = subset.groupby([CASE_COL, column]).size().unstack(fill_value=0)
    counts.columns = [f'cnt_{feature_prefix}__{sanitize_col_name(col)}' for col in counts.columns]
    counts = counts.reset_index()
    base = base.merge(counts, on=CASE_COL, how='left')
    new_cols = [col for col in counts.columns if col != CASE_COL]
    base[new_cols] = base[new_cols].fillna(0).astype(int)
    return base

def repeated_extra(events, column, output_name):
    if column not in events.columns or len(events) == 0:
        return pd.DataFrame({CASE_COL: [], output_name: []})
    temp = events.groupby([CASE_COL, column]).size().rename('count').reset_index()
    temp['extra'] = np.maximum(temp['count'] - 1, 0)
    return temp.groupby(CASE_COL)['extra'].sum().rename(output_name).reset_index()

def infer_feature_types(frame, feature_cols):
    numeric, categorical, diagnostics = ([], [], [])
    for col in feature_cols:
        series = frame[col]
        lower = str(col).lower()
        force_cat = any((token in lower for token in ['department', 'doctype', 'subprocess', 'activity', 'combined', 'resource', 'first_', 'last_']))
        if force_cat:
            categorical.append(col)
            reason = 'forced_process_context'
        elif pd.api.types.is_bool_dtype(series) or pd.api.types.is_numeric_dtype(series):
            numeric.append(col)
            reason = 'numeric_or_bool_dtype'
        else:
            converted = pd.to_numeric(series, errors='coerce')
            share = float(converted.notna().mean())
            if share >= 0.98:
                numeric.append(col)
                reason = f'numeric_like_{share:.2f}'
            else:
                categorical.append(col)
                reason = f'categorical_{share:.2f}'
        diagnostics.append({'feature': col, 'assigned_type': 'numeric' if col in numeric else 'categorical', 'reason': reason})
    return (numeric, categorical, pd.DataFrame(diagnostics))

def make_pipeline(model_name, numeric_cols, categorical_cols):
    transformers = []
    if numeric_cols:
        transformers.append(('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols))
    if categorical_cols:
        transformers.append(('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', make_ohe())]), categorical_cols))
    if not transformers and model_name != 'dummy_prior':
        raise RuntimeError('Keine Features verfügbar.')
    preprocess = ColumnTransformer(transformers=transformers, remainder='drop')
    if model_name == 'dummy_prior':
        model = DummyClassifier(strategy='prior')
    elif model_name == 'logreg_balanced':
        model = LogisticRegression(max_iter=3000, class_weight='balanced', solver='lbfgs', random_state=RANDOM_STATE)
    elif model_name == 'rf_balanced':
        model = RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH, min_samples_leaf=RF_MIN_SAMPLES_LEAF, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=N_JOBS)
    else:
        raise ValueError(model_name)
    return Pipeline([('preprocess', preprocess), ('model', model)])

def model_scores(pipe, X):
    model = pipe.named_steps['model']
    if hasattr(model, 'predict_proba'):
        proba = pipe.predict_proba(X)
        if proba.shape[1] == 1:
            return np.repeat(float(proba[0, 0]), len(X))
        return proba[:, 1]
    if hasattr(model, 'decision_function'):
        raw = pipe.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return pipe.predict(X).astype(float)

def get_pipeline_feature_names(pipe):
    try:
        return list(pipe.named_steps['preprocess'].get_feature_names_out())
    except Exception:
        return []

def bootstrap_ci(y_true, y_score, threshold, repetitions=500, seed=42):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    rng = np.random.default_rng(seed)
    rows = []
    n = len(y_true)
    for _ in range(repetitions):
        idx = rng.integers(0, n, size=n)
        yb, sb = (y_true[idx], y_score[idx])
        if len(np.unique(yb)) < 2:
            continue
        rows.append({'pr_auc': average_precision_score(yb, sb), 'roc_auc': roc_auc_score(yb, sb), 'f1': f1_score(yb, sb >= threshold, zero_division=0), 'precision': precision_score(yb, sb >= threshold, zero_division=0), 'recall': recall_score(yb, sb >= threshold, zero_division=0)})
    boot = pd.DataFrame(rows)
    summary = []
    for metric in ['pr_auc', 'roc_auc', 'f1', 'precision', 'recall']:
        summary.append({'metric': metric, 'estimate': float({'pr_auc': average_precision_score(y_true, y_score), 'roc_auc': roc_auc_score(y_true, y_score), 'f1': f1_score(y_true, y_score >= threshold, zero_division=0), 'precision': precision_score(y_true, y_score >= threshold, zero_division=0), 'recall': recall_score(y_true, y_score >= threshold, zero_division=0)}[metric]), 'ci_lower_95': float(boot[metric].quantile(0.025)) if len(boot) else np.nan, 'ci_upper_95': float(boot[metric].quantile(0.975)) if len(boot) else np.nan, 'valid_bootstrap_repetitions': int(len(boot))})
    return (pd.DataFrame(summary), boot)
print('Helper geladen.')


In [ ]:
# Ereignis und Falldaten laden
print('Lade XES-Log ...')
log = pm4py.read_xes(str(LOG_PATH))
event_df = pm4py.convert_to_dataframe(log)
print('Event DataFrame:', event_df.shape)
CASE_COL = 'case:concept:name' if 'case:concept:name' in event_df.columns else None
ACTIVITY_COL = 'concept:name' if 'concept:name' in event_df.columns else None
RAW_ACTIVITY_COL = 'activity' if 'activity' in event_df.columns else ACTIVITY_COL
TIME_COL = 'time:timestamp' if 'time:timestamp' in event_df.columns else None
ORDER_COL = 'identity:id' if 'identity:id' in event_df.columns else 'eventid' if 'eventid' in event_df.columns else None
RESOURCE_COL = 'org:resource' if 'org:resource' in event_df.columns else None
if CASE_COL is None or ACTIVITY_COL is None or TIME_COL is None:
    raise RuntimeError(f'Kernspalten fehlen: CASE_COL={CASE_COL!r}, ACTIVITY_COL={ACTIVITY_COL!r}, TIME_COL={TIME_COL!r}')
event_df[CASE_COL] = event_df[CASE_COL].astype(str)
event_df[TIME_COL] = pd.to_datetime(event_df[TIME_COL], errors='coerce')
for col in ['doctype', 'subprocess', RAW_ACTIVITY_COL]:
    if col not in event_df.columns:
        event_df[col] = '__missing__'
event_df['combined_activity'] = event_df['doctype'].astype(str) + ' | ' + event_df['subprocess'].astype(str) + ' | ' + event_df[RAW_ACTIVITY_COL].astype(str)
event_df['_is_inspection_context'] = event_df['doctype'].astype(str).str.contains('inspection', case=False, na=False) | event_df['subprocess'].astype(str).str.contains('inspection|on-site|onsite', case=False, na=False, regex=True) | event_df['combined_activity'].astype(str).str.contains('inspection|on-site|onsite', case=False, na=False, regex=True)
event_df['_is_reopened_target_event'] = event_df['subprocess'].astype(str).str.lower().isin(['change', 'objection'])
sort_cols = [CASE_COL, TIME_COL, ORDER_COL if ORDER_COL else ACTIVITY_COL]
event_df = event_df.sort_values(sort_cols, kind='mergesort').reset_index(drop=True)
event_df['_event_pos_in_case'] = event_df.groupby(CASE_COL).cumcount() + 1
event_df['_case_start'] = event_df.groupby(CASE_COL)[TIME_COL].transform('min')
event_df['_elapsed_days'] = (event_df[TIME_COL] - event_df['_case_start']).dt.total_seconds() / 86400
core = pd.read_csv(RECON_CORE_PATH)
core[CASE_COL] = core[CASE_COL].astype(str)
for col in [c for c in core.columns if c.startswith('label_') or c.startswith('selected_') or c.startswith('has_')]:
    core[col] = robust_to_bool(core[col])
core['case_year'] = core['case_year'].astype(str)
core['case_department'] = core['case_department'].fillna('__missing__').astype(str)
case_start = event_df.groupby(CASE_COL)[TIME_COL].min().rename('case_start').reset_index()
first_events = event_df.drop_duplicates(CASE_COL, keep='first').copy()
first_keep = [CASE_COL, 'doctype', 'subprocess', RAW_ACTIVITY_COL]
if RESOURCE_COL is not None:
    first_keep.append(RESOURCE_COL)
first_view = first_events[first_keep].copy()
rename = {'doctype': 'first_doctype', 'subprocess': 'first_subprocess', RAW_ACTIVITY_COL: 'first_activity'}
if RESOURCE_COL is not None:
    rename[RESOURCE_COL] = 'first_resource'
first_view = first_view.rename(columns=rename)
core = core.merge(case_start, on=CASE_COL, how='left').merge(first_view, on=CASE_COL, how='left')
core['case_start_month'] = core['case_start'].dt.month
core['case_start_quarter'] = core['case_start'].dt.quarter
core['case_start_weekday'] = core['case_start'].dt.weekday
first_reopened = event_df[event_df['_is_reopened_target_event']].groupby(CASE_COL).agg(first_reopened_pos=('_event_pos_in_case', 'min'), first_reopened_elapsed_days=('_elapsed_days', 'min'), first_reopened_time=(TIME_COL, 'min')).reset_index()
core = core.merge(first_reopened, on=CASE_COL, how='left')
missing_cases = set(core[CASE_COL]) - set(event_df[CASE_COL].unique())
if missing_cases:
    raise RuntimeError(f'{len(missing_cases)} Cases aus dem Core fehlen im Event Log.')
basic_info = {'events': int(len(event_df)), 'cases_event_log': int(event_df[CASE_COL].nunique()), 'cases_core': int(len(core)), 'timestamp_min': str(event_df[TIME_COL].min()), 'timestamp_max': str(event_df[TIME_COL].max()), 'targets_available': {target: target in core.columns for target in TARGET_CONFIG}}
save_json(basic_info, '00_basic_info.json')
save_csv(core.head(50), '00_reconciliation_core_preview_first50.csv', index=False)
print(json.dumps(basic_info, indent=2, ensure_ascii=False))


In [ ]:
# Population und Split
core = core.sort_values(['case_year', 'case_start', CASE_COL], kind='mergesort').reset_index(drop=True)
core['split'] = 'unused'
cases_2015 = core[core['case_year'].eq('2015')].sort_values(['case_start', CASE_COL], kind='mergesort')
cut = int(len(cases_2015) * TRAIN_SHARE_WITHIN_2015)
train_ids = set(cases_2015.iloc[:cut][CASE_COL])
validation_ids = set(cases_2015.iloc[cut:][CASE_COL])
test_ids = set(core.loc[core['case_year'].eq('2016'), CASE_COL])
sensitivity_2017_ids = set(core.loc[core['case_year'].eq('2017'), CASE_COL])
core.loc[core[CASE_COL].isin(train_ids), 'split'] = 'train'
core.loc[core[CASE_COL].isin(validation_ids), 'split'] = 'validation'
core.loc[core[CASE_COL].isin(test_ids), 'split'] = 'test'
core.loc[core[CASE_COL].isin(sensitivity_2017_ids), 'split'] = 'sensitivity2017'
split_registry = pd.DataFrame([{'split': 'train', 'definition': 'früheste 75 % der 2015er Cases nach case_start', 'n_cases': len(train_ids)}, {'split': 'validation', 'definition': 'späteste 25 % der 2015er Cases nach case_start', 'n_cases': len(validation_ids)}, {'split': 'test', 'definition': 'alle 2016er Cases', 'n_cases': len(test_ids)}, {'split': 'sensitivity2017', 'definition': 'alle 2017er Cases; wegen Rechtszensierung nur Sensitivität', 'n_cases': len(sensitivity_2017_ids)}])
save_csv(split_registry, '01_split_registry.csv', index=False)
prevalence_rows = []
quality_rows = []
active_targets = []
for target, cfg in TARGET_CONFIG.items():
    if target not in core.columns:
        quality_rows.append({'target': target, 'status': 'skip', 'reason': 'Target fehlt im Reconciliation Core'})
        continue
    core[target] = robust_to_bool(core[target])
    split_stats = core[core['split'].isin(['train', 'validation', 'test', 'sensitivity2017'])].groupby('split')[target].agg(['count', 'sum', 'mean']).reset_index()
    split_stats['target'] = target
    split_stats['prevalence_pct'] = split_stats['mean'] * 100
    prevalence_rows.append(split_stats[['target', 'split', 'count', 'sum', 'prevalence_pct']].rename(columns={'count': 'n_cases', 'sum': 'positive_cases'}))
    lookup = split_stats.set_index('split')['sum'].to_dict()
    passes = lookup.get('train', 0) >= MIN_POS_TRAIN and lookup.get('validation', 0) >= MIN_POS_VALIDATION and (lookup.get('test', 0) >= MIN_POS_TEST)
    if cfg.get('optional', False) and (not passes):
        status, reason = ('skip_optional', 'Zu wenige Positivfälle in mindestens einem primären Split')
    elif not passes:
        status, reason = ('fail', 'Primäres Target erfüllt Mindestzahl positiver Fälle nicht')
    else:
        status, reason = ('active', 'Quality Gate bestanden')
        active_targets.append(target)
    quality_rows.append({'target': target, 'status': status, 'reason': reason, **{f'positive_{k}': int(v) for k, v in lookup.items()}})
target_prevalence = pd.concat(prevalence_rows, ignore_index=True) if prevalence_rows else pd.DataFrame()
target_quality = pd.DataFrame(quality_rows)
save_csv(target_prevalence, '02_target_prevalence_by_split.csv', index=False)
save_csv(target_quality, '03_target_quality_gates.csv', index=False)
save_csv(pd.DataFrame([{'scenario': name, **cfg} for name, cfg in SCENARIO_REGISTRY.items()]), '04_feature_scenario_registry.csv', index=False)
display(split_registry)
display(target_prevalence)
display(target_quality)
print('Aktive Targets:', active_targets)
if any((row['status'] == 'fail' for row in quality_rows)):
    raise RuntimeError('Mindestens ein nicht-optionales Target hat das Quality Gate nicht bestanden.')


In [ ]:
# Präfixdatensätze
base_prefix_cache = {}
dataset_cache = {}

def select_window_events(prefix_spec):
    prefix_id = prefix_spec['prefix_id']
    if prefix_id in base_prefix_cache:
        return base_prefix_cache[prefix_id].copy()
    if prefix_spec['prefix_type'] == 'event':
        selected = event_df[event_df['_event_pos_in_case'] <= int(prefix_spec['value'])].copy()
    elif prefix_spec['prefix_type'] == 'time_days':
        selected = event_df[event_df['_elapsed_days'] <= float(prefix_spec['value'])].copy()
    else:
        raise ValueError(prefix_spec)
    base_prefix_cache[prefix_id] = selected.copy()
    return selected

def reopened_at_risk_mask(frame, prefix_spec):
    positive = robust_to_bool(frame['label_reopened_official'])
    if prefix_spec['prefix_type'] == 'event':
        observed = frame['first_reopened_pos'].fillna(np.inf) <= float(prefix_spec['value'])
    else:
        observed = frame['first_reopened_elapsed_days'].fillna(np.inf) <= float(prefix_spec['value'])
    return ~positive | ~observed

def build_modeling_dataset(target, prefix_spec, scenario_name):
    cache_key = (target, prefix_spec['prefix_id'], scenario_name)
    if cache_key in dataset_cache:
        return dataset_cache[cache_key].copy()
    cfg = SCENARIO_REGISTRY[scenario_name]
    base = core[core['split'].isin(['train', 'validation', 'test', 'sensitivity2017'])].copy()
    if cfg.get('at_risk_reopened', False):
        base = base[reopened_at_risk_mask(base, prefix_spec)].copy()
    events = select_window_events(prefix_spec)
    events = events[events[CASE_COL].isin(set(base[CASE_COL]))].copy()
    if cfg.get('remove_inspection_events', False):
        events = events[~events['_is_inspection_context']].copy()
    if cfg.get('remove_reopened_target_events', False):
        events = events[~events['_is_reopened_target_event']].copy()
    feature_df = base[[CASE_COL, 'split', target, 'case_year', 'case_department', 'case_start_month', 'case_start_quarter', 'case_start_weekday', 'selected_random', 'selected_risk', 'selected_manually', 'selected_any_inspection', 'has_inspection_event_context']].copy()
    counts = events.groupby(CASE_COL).size().rename('prefix_event_count').reset_index()
    feature_df = feature_df.merge(counts, on=CASE_COL, how='left')
    feature_df['prefix_event_count'] = feature_df['prefix_event_count'].fillna(0).astype(int)
    if len(events):
        times = events.groupby(CASE_COL)[TIME_COL].agg(prefix_start='min', prefix_last='max').reset_index()
        times['prefix_observed_duration_hours'] = (times['prefix_last'] - times['prefix_start']).dt.total_seconds() / 3600
        feature_df = feature_df.merge(times[[CASE_COL, 'prefix_observed_duration_hours']], on=CASE_COL, how='left')
    feature_df['prefix_observed_duration_hours'] = feature_df.get('prefix_observed_duration_hours', 0)
    feature_df['prefix_observed_duration_hours'] = pd.to_numeric(feature_df['prefix_observed_duration_hours'], errors='coerce').fillna(0)
    for col, output in [(RAW_ACTIVITY_COL, 'prefix_n_activities'), ('combined_activity', 'prefix_n_combined_activities'), ('subprocess', 'prefix_n_subprocesses'), ('doctype', 'prefix_n_doctypes')]:
        temp = events.groupby(CASE_COL)[col].nunique().rename(output).reset_index() if len(events) else pd.DataFrame({CASE_COL: [], output: []})
        feature_df = feature_df.merge(temp, on=CASE_COL, how='left')
        feature_df[output] = feature_df[output].fillna(0).astype(int)
    if RESOURCE_COL is not None:
        temp = events.groupby(CASE_COL)[RESOURCE_COL].nunique().rename('prefix_n_resources').reset_index() if len(events) else pd.DataFrame({CASE_COL: [], 'prefix_n_resources': []})
        feature_df = feature_df.merge(temp, on=CASE_COL, how='left')
        feature_df['prefix_n_resources'] = feature_df['prefix_n_resources'].fillna(0).astype(int)
    else:
        feature_df['prefix_n_resources'] = 0
    feature_df = feature_df.merge(repeated_extra(events, RAW_ACTIVITY_COL, 'prefix_raw_rework_extra'), on=CASE_COL, how='left')
    feature_df = feature_df.merge(repeated_extra(events, 'combined_activity', 'prefix_combined_rework_extra'), on=CASE_COL, how='left')
    feature_df[['prefix_raw_rework_extra', 'prefix_combined_rework_extra']] = feature_df[['prefix_raw_rework_extra', 'prefix_combined_rework_extra']].fillna(0).astype(int)

    def add_any(mask, name):
        nonlocal feature_df
        if len(events):
            temp = mask.groupby(events[CASE_COL]).any().rename(name).reset_index()
            feature_df = feature_df.merge(temp, on=CASE_COL, how='left')
        else:
            feature_df[name] = False
        feature_df[name] = robust_to_bool(feature_df[name])
    add_any(events['_is_inspection_context'] if len(events) else pd.Series(dtype=bool), 'prefix_has_inspection_context')
    add_any(events[RAW_ACTIVITY_COL].astype(str).str.lower().eq('remove document') if len(events) else pd.Series(dtype=bool), 'prefix_has_remove_document')
    add_any(events[RAW_ACTIVITY_COL].astype(str).str.contains('payment', case=False, na=False) if len(events) else pd.Series(dtype=bool), 'prefix_has_payment_activity')
    add_any(events[RAW_ACTIVITY_COL].astype(str).str.contains('decide|decision', case=False, na=False, regex=True) if len(events) else pd.Series(dtype=bool), 'prefix_has_decision_activity')
    add_any(events['_is_reopened_target_event'] if len(events) else pd.Series(dtype=bool), 'prefix_has_change_or_objection')
    if len(events):
        context_cols = [CASE_COL, RAW_ACTIVITY_COL, 'subprocess', 'doctype'] + ([RESOURCE_COL] if RESOURCE_COL else [])
        first = events.drop_duplicates(CASE_COL, keep='first')[context_cols].copy()
        rename_first = {RAW_ACTIVITY_COL: 'first_activity_prefix', 'subprocess': 'first_subprocess_prefix', 'doctype': 'first_doctype_prefix'}
        if RESOURCE_COL:
            rename_first[RESOURCE_COL] = 'first_resource_prefix'
        feature_df = feature_df.merge(first.rename(columns=rename_first), on=CASE_COL, how='left')
    for col in ['first_activity_prefix', 'first_subprocess_prefix', 'first_doctype_prefix', 'first_resource_prefix']:
        if col not in feature_df.columns:
            feature_df[col] = '__none__'
        feature_df[col] = feature_df[col].fillna('__none__').astype(str)
    if len(events):
        last_cols = [CASE_COL, RAW_ACTIVITY_COL, 'subprocess', 'doctype'] + ([RESOURCE_COL] if RESOURCE_COL else [])
        last = events.groupby(CASE_COL, sort=False).last().reset_index()[last_cols]
        rename = {RAW_ACTIVITY_COL: 'last_activity_prefix', 'subprocess': 'last_subprocess_prefix', 'doctype': 'last_doctype_prefix'}
        if RESOURCE_COL:
            rename[RESOURCE_COL] = 'last_resource_prefix'
        feature_df = feature_df.merge(last.rename(columns=rename), on=CASE_COL, how='left')
    for col in ['last_activity_prefix', 'last_subprocess_prefix', 'last_doctype_prefix', 'last_resource_prefix']:
        if col not in feature_df.columns:
            feature_df[col] = '__none__'
        feature_df[col] = feature_df[col].fillna('__none__').astype(str)
    train_cases_local = set(feature_df.loc[feature_df['split'].eq('train'), CASE_COL])
    vocab = {'activity': top_values(events, RAW_ACTIVITY_COL, train_cases_local, TOP_N_RAW_ACTIVITIES), 'combined': top_values(events, 'combined_activity', train_cases_local, TOP_N_COMBINED_ACTIVITIES), 'subprocess': top_values(events, 'subprocess', train_cases_local, TOP_N_SUBPROCESSES), 'doctype': top_values(events, 'doctype', train_cases_local, TOP_N_DOCTYPES), 'resource': top_values(events, RESOURCE_COL, train_cases_local, TOP_N_RESOURCES) if RESOURCE_COL else []}
    feature_df = add_count_features(feature_df, events, RAW_ACTIVITY_COL, vocab['activity'], 'activity')
    feature_df = add_count_features(feature_df, events, 'combined_activity', vocab['combined'], 'combined')
    feature_df = add_count_features(feature_df, events, 'subprocess', vocab['subprocess'], 'subprocess')
    feature_df = add_count_features(feature_df, events, 'doctype', vocab['doctype'], 'doctype')
    if RESOURCE_COL:
        feature_df = add_count_features(feature_df, events, RESOURCE_COL, vocab['resource'], 'resource')
    for pattern in cfg.get('drop_feature_patterns', []):
        drop_cols = [col for col in feature_df.columns if pattern.lower() in str(col).lower() and col not in [CASE_COL, 'split', target]]
        feature_df = feature_df.drop(columns=drop_cols, errors='ignore')
    feature_df = feature_df.drop(columns=['case_year'], errors='ignore')
    feature_df = feature_df.drop(columns=['has_inspection_event_context'], errors='ignore')
    for col in feature_df.columns:
        if col in [CASE_COL, 'split', target]:
            continue
        if pd.api.types.is_bool_dtype(feature_df[col]) or col.startswith('prefix_has_') or col.startswith('selected_'):
            feature_df[col] = robust_to_bool(feature_df[col]).astype(int)
        elif pd.api.types.is_object_dtype(feature_df[col]) or pd.api.types.is_string_dtype(feature_df[col]):
            feature_df[col] = feature_df[col].fillna('__missing__').astype(str)
    feature_df['_prefix_id'] = prefix_spec['prefix_id']
    feature_df['_scenario'] = scenario_name
    dataset_cache[cache_key] = feature_df.copy()
    return feature_df
eligibility_rows = []
for target in active_targets:
    cfg = TARGET_CONFIG[target]
    scenarios = [cfg['primary_scenario']] + cfg.get('sensitivity_scenarios', [])
    for scenario in scenarios:
        for prefix in PREFIX_SPECS:
            dataset = build_modeling_dataset(target, prefix, scenario)
            for split in ['train', 'validation', 'test', 'sensitivity2017']:
                sub = dataset[dataset['split'].eq(split)]
                eligibility_rows.append({'target': target, 'scenario': scenario, 'prefix_id': prefix['prefix_id'], 'split': split, 'n_cases': len(sub), 'positive_cases': int(robust_to_bool(sub[target]).sum()) if len(sub) else 0, 'prevalence_pct': float(robust_to_bool(sub[target]).mean() * 100) if len(sub) else np.nan})
eligibility_df = pd.DataFrame(eligibility_rows)
save_csv(eligibility_df, '05_target_prefix_eligibility.csv', index=False)
display(eligibility_df.head(20))


In [ ]:
# Modelle auswerten
META_COLS = {CASE_COL, 'split', '_prefix_id', '_scenario'}

def feature_columns(frame, target):
    forbidden_exact = META_COLS | {target, 'case_year', 'case_start', 'case_end', 'duration_days', 'event_count', 'combined_rework_extra', 'event_count_no_inspection', 'rework_no_inspection', 'first_reopened_pos', 'first_reopened_elapsed_days', 'first_reopened_time'}
    cols = []
    for col in frame.columns:
        if col in forbidden_exact:
            continue
        if str(col).lower().startswith('label_'):
            continue
        cols.append(col)
    return cols

def fit_one(frame, target, prefix_id, scenario, model_name):
    frame = frame.copy()
    frame[target] = robust_to_bool(frame[target])
    features = feature_columns(frame, target)
    numeric, categorical, diagnostics = infer_feature_types(frame, features)
    pipe = make_pipeline(model_name, numeric, categorical)
    parts = {}
    for split in ['train', 'validation', 'test', 'sensitivity2017']:
        sub = frame[frame['split'].eq(split)].copy()
        parts[split] = {'X': sub[features].copy(), 'y': robust_to_bool(sub[target]).astype(int).to_numpy(), 'meta': sub[[CASE_COL, 'split', target]].copy()}
    if len(parts['train']['y']) == 0 or len(np.unique(parts['train']['y'])) < 2:
        raise RuntimeError('Train-Split leer oder nur eine Klasse.')
    if len(parts['validation']['y']) == 0 or len(np.unique(parts['validation']['y'])) < 2:
        raise RuntimeError('Validation-Split leer oder nur eine Klasse.')
    if len(parts['test']['y']) == 0 or len(np.unique(parts['test']['y'])) < 2:
        raise RuntimeError('Test-Split leer oder nur eine Klasse.')
    pipe.fit(parts['train']['X'], parts['train']['y'])
    scores = {split: model_scores(pipe, part['X']) for split, part in parts.items() if len(part['X'])}
    threshold, validation_best_f1 = select_threshold_by_f1(parts['validation']['y'], scores['validation'])
    eval_rows = []
    for split in ['train', 'validation', 'test', 'sensitivity2017']:
        if split not in scores:
            continue
        for policy, used_threshold in [('fixed_0_5', 0.5), ('validation_f1_threshold', threshold)]:
            eval_rows.append(evaluate_scores(parts[split]['y'], scores[split], used_threshold, {'target': target, 'prefix_id': prefix_id, 'scenario': scenario, 'model': model_name, 'split': split, 'threshold_policy': policy, 'validation_selected_threshold': threshold, 'validation_best_f1': validation_best_f1, 'raw_feature_count': len(features), 'numeric_feature_count': len(numeric), 'categorical_feature_count': len(categorical)}))
    predictions = []
    for split in ['validation', 'test', 'sensitivity2017']:
        if split not in scores:
            continue
        temp = parts[split]['meta'].copy()
        temp['target'] = target
        temp['prefix_id'] = prefix_id
        temp['scenario'] = scenario
        temp['model'] = model_name
        temp['y_true'] = parts[split]['y']
        temp['score'] = scores[split]
        temp['threshold'] = threshold
        temp['pred_validation_threshold'] = (scores[split] >= threshold).astype(int)
        predictions.append(temp)
    predictions = pd.concat(predictions, ignore_index=True)
    importance = pd.DataFrame()
    names = get_pipeline_feature_names(pipe)
    model = pipe.named_steps['model']
    if model_name == 'logreg_balanced' and hasattr(model, 'coef_') and (len(names) == len(model.coef_[0])):
        importance = pd.DataFrame({'feature': names, 'importance': model.coef_[0]})
    elif model_name == 'rf_balanced' and hasattr(model, 'feature_importances_') and (len(names) == len(model.feature_importances_)):
        importance = pd.DataFrame({'feature': names, 'importance': model.feature_importances_})
    if len(importance):
        importance['abs_importance'] = importance['importance'].abs()
        importance['target'] = target
        importance['prefix_id'] = prefix_id
        importance['scenario'] = scenario
        importance['model'] = model_name
    diagnostics['target'] = target
    diagnostics['prefix_id'] = prefix_id
    diagnostics['scenario'] = scenario
    diagnostics['model'] = model_name
    return {'pipeline': pipe, 'features': features, 'eval': pd.DataFrame(eval_rows), 'predictions': predictions, 'importance': importance, 'feature_diagnostics': diagnostics, 'parts': parts}
all_eval, all_predictions, all_importance, all_feature_diag, failures = ([], [], [], [], [])
trained = {}
for target in active_targets:
    target_cfg = TARGET_CONFIG[target]
    scenarios = [target_cfg['primary_scenario']] + target_cfg.get('sensitivity_scenarios', [])
    for scenario in scenarios:
        if target_cfg.get('optional', False):
            prefixes = [p for p in PREFIX_SPECS if p['prefix_id'] in ['time90d', 'time120d']]
            models_for_run = ['dummy_prior', 'logreg_balanced', 'rf_balanced']
        else:
            prefixes = PREFIX_SPECS if scenario == target_cfg['primary_scenario'] else [p for p in PREFIX_SPECS if p['prefix_id'] in ['event20', 'time90d', 'time120d']]
            models_for_run = MODEL_NAMES
        for prefix in prefixes:
            dataset = build_modeling_dataset(target, prefix, scenario)
            for model_name in models_for_run:
                print(f"Trainiere {target} | {scenario} | {prefix['prefix_id']} | {model_name}")
                try:
                    result = fit_one(dataset, target, prefix['prefix_id'], scenario, model_name)
                    key = (target, scenario, prefix['prefix_id'], model_name)
                    trained[key] = result
                    all_eval.append(result['eval'])
                    all_predictions.append(result['predictions'])
                    if len(result['importance']):
                        all_importance.append(result['importance'].head(150))
                    all_feature_diag.append(result['feature_diagnostics'])
                except Exception as exc:
                    failures.append({'target': target, 'scenario': scenario, 'prefix_id': prefix['prefix_id'], 'model': model_name, 'error': repr(exc)})
                    print('  FEHLER:', repr(exc))
evaluation_df = pd.concat(all_eval, ignore_index=True) if all_eval else pd.DataFrame()
prediction_df = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()
importance_df = pd.concat(all_importance, ignore_index=True) if all_importance else pd.DataFrame()
feature_diagnostics_df = pd.concat(all_feature_diag, ignore_index=True) if all_feature_diag else pd.DataFrame()
failure_df = pd.DataFrame(failures)
save_csv(evaluation_df, '06_all_model_evaluation.csv', index=False)
save_csv(prediction_df, '07_validation_test_predictions.csv', index=False)
save_csv(importance_df, '08_model_feature_importances.csv', index=False)
save_csv(feature_diagnostics_df, '09_feature_type_diagnostics.csv', index=False)
save_csv(failure_df, '10_model_failure_log.csv', index=False)
print('Evaluation rows:', len(evaluation_df), 'Failures:', len(failure_df))
if len(evaluation_df) == 0:
    display(failure_df)
    raise RuntimeError('Kein Modell erfolgreich. Failure Log prüfen.')


In [ ]:
# Modellauswahl
primary_scenarios = {target: cfg['primary_scenario'] for target, cfg in TARGET_CONFIG.items() if target in active_targets}
validation_candidates = evaluation_df[evaluation_df['split'].eq('validation') & evaluation_df['threshold_policy'].eq('fixed_0_5') & evaluation_df.apply(lambda row: primary_scenarios.get(row['target']) == row['scenario'], axis=1)].copy()
non_dummy = validation_candidates[~validation_candidates['model'].eq('dummy_prior')].copy()
selected_rows = []
for target in active_targets:
    candidates = non_dummy[non_dummy['target'].eq(target)].sort_values(['pr_auc_average_precision', 'roc_auc'], ascending=[False, False])
    if len(candidates) == 0:
        candidates = validation_candidates[validation_candidates['target'].eq(target)].sort_values('pr_auc_average_precision', ascending=False)
    if len(candidates) == 0:
        continue
    chosen = candidates.iloc[0].to_dict()
    chosen['selection_rule'] = 'max validation PR-AUC; ROC-AUC tie-break; test not used'
    selected_rows.append(chosen)
selected_validation_df = pd.DataFrame(selected_rows)
selected_test_rows = []
for _, row in selected_validation_df.iterrows():
    match = evaluation_df[evaluation_df['target'].eq(row['target']) & evaluation_df['scenario'].eq(row['scenario']) & evaluation_df['prefix_id'].eq(row['prefix_id']) & evaluation_df['model'].eq(row['model']) & evaluation_df['split'].eq('test') & evaluation_df['threshold_policy'].eq('validation_f1_threshold')]
    if len(match):
        selected_test_rows.append(match.iloc[0].to_dict())
selected_test_df = pd.DataFrame(selected_test_rows)
save_csv(validation_candidates, '11_validation_model_candidates.csv', index=False)
save_csv(selected_validation_df, '12_validation_selected_models.csv', index=False)
save_csv(selected_test_df, '13_selected_models_test_results.csv', index=False)
display(selected_validation_df[['target', 'scenario', 'prefix_id', 'model', 'pr_auc_average_precision', 'roc_auc', 'f1']])
display(selected_test_df[['target', 'scenario', 'prefix_id', 'model', 'pr_auc_average_precision', 'roc_auc', 'precision', 'recall', 'f1', 'brier_score']])


In [ ]:
# Bootstrap und Merkmalsbedeutung
bootstrap_summaries = []
calibration_rows = []
permutation_rows = []
for _, selected in selected_test_df.iterrows():
    target, scenario, prefix_id, model_name = (selected['target'], selected['scenario'], selected['prefix_id'], selected['model'])
    preds = prediction_df[prediction_df['target'].eq(target) & prediction_df['scenario'].eq(scenario) & prediction_df['prefix_id'].eq(prefix_id) & prediction_df['model'].eq(model_name) & prediction_df['split'].eq('test')].copy()
    if len(preds) == 0:
        continue
    threshold = float(preds['threshold'].iloc[0])
    ci, _ = bootstrap_ci(preds['y_true'], preds['score'], threshold, BOOTSTRAP_REPETITIONS, RANDOM_STATE)
    ci['target'] = target
    ci['scenario'] = scenario
    ci['prefix_id'] = prefix_id
    ci['model'] = model_name
    bootstrap_summaries.append(ci)
    try:
        bins = pd.qcut(preds['score'], q=10, duplicates='drop')
        calib = preds.assign(bin=bins).groupby('bin', observed=True).agg(mean_predicted_score=('score', 'mean'), observed_positive_rate=('y_true', 'mean'), n=('y_true', 'count')).reset_index(drop=True)
        calib['target'] = target
        calib['scenario'] = scenario
        calib['prefix_id'] = prefix_id
        calib['model'] = model_name
        calibration_rows.append(calib)
    except Exception as exc:
        analysis_notes.append(f'Calibration failed {target}: {repr(exc)}')
    if RUN_PERMUTATION_IMPORTANCE and model_name != 'dummy_prior':
        key = (target, scenario, prefix_id, model_name)
        result = trained.get(key)
        if result is not None:
            X = result['parts']['test']['X']
            y = result['parts']['test']['y']
            if len(X) > PERMUTATION_SAMPLE_N:
                rng = np.random.default_rng(RANDOM_STATE)
                idx = rng.choice(len(X), size=PERMUTATION_SAMPLE_N, replace=False)
                Xp, yp = (X.iloc[idx].copy(), y[idx])
            else:
                Xp, yp = (X.copy(), y)
            try:
                perm = permutation_importance(result['pipeline'], Xp, yp, scoring='average_precision', n_repeats=PERMUTATION_REPEATS, random_state=RANDOM_STATE, n_jobs=N_JOBS)
                temp = pd.DataFrame({'raw_feature': result['features'], 'importance_mean': perm.importances_mean, 'importance_std': perm.importances_std}).sort_values('importance_mean', ascending=False)
                temp['target'] = target
                temp['scenario'] = scenario
                temp['prefix_id'] = prefix_id
                temp['model'] = model_name
                permutation_rows.append(temp)
            except Exception as exc:
                analysis_notes.append(f'Permutation importance failed {target}: {repr(exc)}')
bootstrap_df = pd.concat(bootstrap_summaries, ignore_index=True) if bootstrap_summaries else pd.DataFrame()
calibration_df = pd.concat(calibration_rows, ignore_index=True) if calibration_rows else pd.DataFrame()
permutation_df = pd.concat(permutation_rows, ignore_index=True) if permutation_rows else pd.DataFrame()
save_csv(bootstrap_df, '14_selected_models_bootstrap_ci.csv', index=False)
save_csv(calibration_df, '15_selected_models_calibration.csv', index=False)
save_csv(permutation_df, '16_selected_models_permutation_importance.csv', index=False)
display(bootstrap_df)


In [ ]:
# Zeitfenster und Ablation
comparison_rows = []
for target in active_targets:
    primary = TARGET_CONFIG[target]['primary_scenario']
    for model_name in ['logreg_balanced', 'rf_balanced']:
        rows = evaluation_df[evaluation_df['target'].eq(target) & evaluation_df['scenario'].eq(primary) & evaluation_df['model'].eq(model_name) & evaluation_df['split'].isin(['validation', 'test']) & evaluation_df['threshold_policy'].eq('fixed_0_5') & evaluation_df['prefix_id'].isin(['time90d', 'time120d'])]
        for split in ['validation', 'test']:
            pivot = rows[rows['split'].eq(split)].set_index('prefix_id')
            if {'time90d', 'time120d'}.issubset(pivot.index):
                comparison_rows.append({'target': target, 'scenario': primary, 'model': model_name, 'split': split, 'pr_auc_90': pivot.loc['time90d', 'pr_auc_average_precision'], 'pr_auc_120': pivot.loc['time120d', 'pr_auc_average_precision'], 'delta_pr_auc_120_minus_90': pivot.loc['time120d', 'pr_auc_average_precision'] - pivot.loc['time90d', 'pr_auc_average_precision'], 'roc_auc_90': pivot.loc['time90d', 'roc_auc'], 'roc_auc_120': pivot.loc['time120d', 'roc_auc'], 'delta_roc_auc_120_minus_90': pivot.loc['time120d', 'roc_auc'] - pivot.loc['time90d', 'roc_auc']})
comparison_90_120 = pd.DataFrame(comparison_rows)
save_csv(comparison_90_120, '17_time90_vs_time120_comparison.csv', index=False)
ablation_rows = []
for target in ['label_scd_p90_or_global', 'label_scd_no_inspection_p90_or']:
    if target not in active_targets:
        continue
    primary = TARGET_CONFIG[target]['primary_scenario']
    for sensitivity in TARGET_CONFIG[target].get('sensitivity_scenarios', []):
        for prefix_id in ['event20', 'time90d', 'time120d']:
            for model_name in ['logreg_balanced', 'rf_balanced']:
                base = evaluation_df[evaluation_df['target'].eq(target) & evaluation_df['scenario'].eq(primary) & evaluation_df['prefix_id'].eq(prefix_id) & evaluation_df['model'].eq(model_name) & evaluation_df['split'].eq('test') & evaluation_df['threshold_policy'].eq('fixed_0_5')]
                alt = evaluation_df[evaluation_df['target'].eq(target) & evaluation_df['scenario'].eq(sensitivity) & evaluation_df['prefix_id'].eq(prefix_id) & evaluation_df['model'].eq(model_name) & evaluation_df['split'].eq('test') & evaluation_df['threshold_policy'].eq('fixed_0_5')]
                if len(base) and len(alt):
                    ablation_rows.append({'target': target, 'primary_scenario': primary, 'sensitivity_scenario': sensitivity, 'prefix_id': prefix_id, 'model': model_name, 'primary_pr_auc': base.iloc[0]['pr_auc_average_precision'], 'sensitivity_pr_auc': alt.iloc[0]['pr_auc_average_precision'], 'delta_pr_auc_sensitivity_minus_primary': alt.iloc[0]['pr_auc_average_precision'] - base.iloc[0]['pr_auc_average_precision'], 'primary_roc_auc': base.iloc[0]['roc_auc'], 'sensitivity_roc_auc': alt.iloc[0]['roc_auc']})
ablation_df = pd.DataFrame(ablation_rows)
save_csv(ablation_df, '18_target_specific_ablation_effects.csv', index=False)
display(comparison_90_120)
display(ablation_df.head(20))


In [ ]:
# Fehler und Sensitivitätsanalyse
error_rows = []
sensitivity_rows = []
for _, selected in selected_test_df.iterrows():
    target, scenario, prefix_id, model_name = (selected['target'], selected['scenario'], selected['prefix_id'], selected['model'])
    preds = prediction_df[prediction_df['target'].eq(target) & prediction_df['scenario'].eq(scenario) & prediction_df['prefix_id'].eq(prefix_id) & prediction_df['model'].eq(model_name)].merge(core[[CASE_COL, 'case_department', 'has_inspection_event_context', 'selected_random', 'selected_risk', 'selected_manually']], on=CASE_COL, how='left')
    for split in ['test', 'sensitivity2017']:
        sub_split = preds[preds['split'].eq(split)].copy()
        if len(sub_split) == 0:
            continue
        if split == 'sensitivity2017' and (not (RUN_2017_SENSITIVITY and TARGET_CONFIG[target].get('allow_2017_sensitivity', False))):
            continue
        threshold = float(sub_split['threshold'].iloc[0])
        sub_split['pred'] = sub_split['score'] >= threshold
        sensitivity_rows.append(evaluate_scores(sub_split['y_true'], sub_split['score'], threshold, {'target': target, 'scenario': scenario, 'prefix_id': prefix_id, 'model': model_name, 'split': split, 'threshold_policy': 'validation_f1_threshold'}))
        for group_col in ['case_department', 'has_inspection_event_context']:
            for group_value, group in sub_split.groupby(group_col, dropna=False):
                if len(group) < 30:
                    continue
                tn, fp, fn, tp = confusion_matrix(group['y_true'], group['pred'], labels=[0, 1]).ravel()
                error_rows.append({'target': target, 'split': split, 'group_variable': group_col, 'group_value': str(group_value), 'n_cases': len(group), 'positive_cases': int(group['y_true'].sum()), 'prevalence_pct': float(group['y_true'].mean() * 100), 'mean_score': float(group['score'].mean()), 'false_positive_rate': float(fp / (fp + tn)) if fp + tn else np.nan, 'false_negative_rate': float(fn / (fn + tp)) if fn + tp else np.nan, 'precision': float(precision_score(group['y_true'], group['pred'], zero_division=0)), 'recall': float(recall_score(group['y_true'], group['pred'], zero_division=0))})
error_analysis_df = pd.DataFrame(error_rows)
sensitivity_2017_df = pd.DataFrame(sensitivity_rows)
save_csv(error_analysis_df, '19_selected_models_group_error_analysis.csv', index=False)
save_csv(sensitivity_2017_df, '20_selected_models_2017_sensitivity.csv', index=False)
display(error_analysis_df.head(30))
display(sensitivity_2017_df)


In [ ]:
# Abbildungen
val_plot = validation_candidates.copy()
fig, ax = plt.subplots(figsize=(11, 6))
for (target, model), group in val_plot.groupby(['target', 'model']):
    order = [p['prefix_id'] for p in PREFIX_SPECS]
    group = group.set_index('prefix_id').reindex(order).reset_index()
    ax.plot(group['prefix_id'], group['pr_auc_average_precision'], marker='o', label=f"{TARGET_CONFIG[target]['short_name']} | {model}")
ax.set_ylabel('Validation PR-AUC')
ax.set_xlabel('Prefix')
ax.set_title('Validation-Leistung nach Target, Prefix und Modell')
ax.tick_params(axis='x', rotation=30)
ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=2)
save_fig(fig, 'fig_01_validation_pr_auc_by_target_prefix.png')
if len(bootstrap_df):
    pr = bootstrap_df[bootstrap_df['metric'].eq('pr_auc')].copy()
    pr['target_short'] = pr['target'].map(lambda x: TARGET_CONFIG[x]['short_name'])
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(pr))
    ax.bar(x, pr['estimate'])
    ax.errorbar(x, pr['estimate'], yerr=[pr['estimate'] - pr['ci_lower_95'], pr['ci_upper_95'] - pr['estimate']], fmt='none', capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(pr['target_short'], rotation=20)
    ax.set_ylabel('Test PR-AUC')
    ax.set_title('Validation-ausgewählte Modelle mit 95-%-Bootstrap-CI')
    ax.grid(axis='y', alpha=0.3)
    save_fig(fig, 'fig_02_selected_test_pr_auc_ci.png')
if len(comparison_90_120):
    subset = comparison_90_120[comparison_90_120['split'].eq('validation') & comparison_90_120['model'].eq('rf_balanced')].copy()
    subset['target_short'] = subset['target'].map(lambda x: TARGET_CONFIG[x]['short_name'])
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(subset))
    width = 0.35
    ax.bar(x - width / 2, subset['pr_auc_90'], width, label='90 Tage')
    ax.bar(x + width / 2, subset['pr_auc_120'], width, label='120 Tage')
    ax.set_xticks(x)
    ax.set_xticklabels(subset['target_short'], rotation=20)
    ax.set_ylabel('Validation PR-AUC')
    ax.set_title('Zusatznutzen von 120 gegenüber 90 Tagen')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    save_fig(fig, 'fig_03_time90_vs_time120.png')
if len(calibration_df):
    fig, ax = plt.subplots(figsize=(7, 6))
    for target, group in calibration_df.groupby('target'):
        ax.plot(group['mean_predicted_score'], group['observed_positive_rate'], marker='o', label=TARGET_CONFIG[target]['short_name'])
    ax.plot([0, 1], [0, 1], linestyle='--')
    ax.set_xlabel('Mittlerer vorhergesagter Score')
    ax.set_ylabel('Beobachtete Positivrate')
    ax.set_title('Calibration der ausgewählten Modelle')
    ax.legend()
    ax.grid(alpha=0.3)
    save_fig(fig, 'fig_04_selected_model_calibration.png')
if len(error_analysis_df):
    insp = error_analysis_df[error_analysis_df['split'].eq('test') & error_analysis_df['group_variable'].eq('has_inspection_event_context')].copy()
    if len(insp):
        insp['target_short'] = insp['target'].map(lambda x: TARGET_CONFIG[x]['short_name'])
        labels = insp['target_short'] + ' | insp=' + insp['group_value']
        fig, ax = plt.subplots(figsize=(10, 5))
        x = np.arange(len(insp))
        width = 0.35
        ax.bar(x - width / 2, insp['false_positive_rate'], width, label='FPR')
        ax.bar(x + width / 2, insp['false_negative_rate'], width, label='FNR')
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=30, ha='right')
        ax.set_ylabel('Rate')
        ax.set_title('Fehler nach Inspection-Kontext')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        save_fig(fig, 'fig_05_error_rates_by_inspection.png')
print('Abbildungen erstellt:', len(created_figures))


In [ ]:
# Qualitätsprüfungen
quality_gate_rows = []
for target in active_targets:
    selected_target = selected_test_df[selected_test_df['target'].eq(target)]
    quality_gate_rows.extend([{'target': target, 'gate': 'Mindestfallzahlen', 'status': 'PASS' if target_quality.loc[target_quality['target'].eq(target), 'status'].iloc[0] == 'active' else 'REVIEW', 'evidence': '03_target_quality_gates.csv'}, {'target': target, 'gate': 'Validation-basierte Modellauswahl', 'status': 'PASS' if len(selected_target) else 'FAIL', 'evidence': '12_validation_selected_models.csv'}, {'target': target, 'gate': 'Bootstrap-CI', 'status': 'PASS' if len(bootstrap_df[bootstrap_df['target'].eq(target)]) else 'REVIEW', 'evidence': '14_selected_models_bootstrap_ci.csv'}, {'target': target, 'gate': 'Target-spezifische Leakage-Kontrolle', 'status': 'PASS', 'evidence': TARGET_CONFIG[target]['primary_scenario']}])
quality_gate_df = pd.DataFrame(quality_gate_rows)
save_csv(quality_gate_df, '22_quality_gate_register.csv', index=False)
selected_summary = selected_test_df.merge(bootstrap_df[bootstrap_df['metric'].eq('pr_auc')][['target', 'scenario', 'prefix_id', 'model', 'ci_lower_95', 'ci_upper_95']], on=['target', 'scenario', 'prefix_id', 'model'], how='left')
save_csv(selected_summary, '24_selected_models_summary_for_thesis.csv', index=False)
display(quality_gate_df)
display(selected_summary)
